# Experiment 021: from-scratch 1000-step schedule-regime control (Colab)

Fresh grpo on the same merged gsm8k base each 006 run used, one smooth
1000-step schedule, compared against the observed 500+500 restarted regime.
This compares two complete schedule regimes, the smooth one carries approximately 33% more
cumulative learning-rate exposure, so restart causality is not isolated and
no reading claims it. Full rules in the 021 plan.

Staged rollout: the first pass runs the pilot arm (seed 1). After I'll do a technical
check, I'll rerun the same notebook with the seed list widened to all three,
finished arms skip. The continuation is bug-gated, never result-gated: the
remaining arms run whatever the pilot's scientific outcome is, only a
technical failure pauses for a fix.

Fresh runs mean the sentinel is active again. Hardware: a100-40gb. ~9h per arm.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [ ]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before =  472.47   # fill the current balance
unit_rate = 5.3   # units per hour from the panel
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '021_scratch', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260729_103627 | cu_before = 472.47
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: any A100, 40GB preferred

In [ ]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
assert 'A100' in gpu, 'pick an a100 runtime'
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram {total_gb:.1f} GiB' + ('' if total_gb < 60 else ' (80gb variant, acceptable, gets recorded)'))

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-a93bd1b9-544d-2f39-8c4a-1f442abd83ca)
total vram 39.5 GiB


## 2. Clone or update the repo (automatic)

In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # instead of using the stale clone
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 152 (delta 140), reused 128 (delta 116), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 1.37 MiB | 37.84 MiB/s, done.
Resolving deltas: 100% (140/140), completed with 73 local objects.
From https://github.com/oleh-hastov/INM363-IndividualProject-Dissertation
   223adf9..7f830ee  main       -> origin/main
Updating 223adf9..7f830ee
Fast-forward
 001_Code/analysis/aux_train_check.py               |  4 +-
 001_Code/analysis/stats.py                         | 14 ++--
 001_Code/checkpoint.py                             |  4 +-
 001_Code/config.py                                 | 16 ++--
 001_Code/configs/001_colab_full_b05.yaml           | 14 ++--
 001_Code/configs/002_colab_full_b05.yaml           |  4 +-
 001_Code/configs/003_colab_full_b3_seed0.yaml      |  8 +-
 001_Code/configs/003_colab_full_b3_seed1.yaml      | 10 +--
 .../c

## 3. Install dependencies

In [ ]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


deps installed


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [ ]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [ ]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 10, 'need some drive headroom for the continuation artifacts'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260729_103627.json
drive free: 177 GB


## 6. Arms for this pass

First pass: the pilot arm only. Second pass: widen to (1, 0, 2), finished
arms skip automatically.

In [ ]:
SEEDS_TO_RUN = (1, 0, 2)   # second pass, finished arms skip and get reverified
RECOVERED = {1: '018_ext_B05_seed1_step1000', 2: '019_ext_B05_seed2_step1000', 0: '020_ext_B05_seed0_step1000'}
print('arms this pass:', SEEDS_TO_RUN)

arms this pass: (1, 0, 2)


## 7. Seed the arm directories (merged base only, no checkpoint)

Fresh runs: each arm gets a hash-verified copy of its 006 counterpart's
merged base and a config hash written in the same transaction. A directory
with a manifest is verified for resume, a directory without one must be
empty.

In [ ]:
import hashlib, json, os, shutil, glob, sys, time
sys.path.insert(0, os.getcwd())
from config import load_config
from checkpoint import write_config_hash, _config_hash

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
for s in SEEDS_TO_RUN:
    SRC = os.path.join(D, 'checkpoints', f'006_COLAB_FULL_B05_sft_grpo_seed{s}')
    DST = os.path.join(D, 'checkpoints', f'021_scratch_B05_seed{s}')
    res = os.path.join(D, 'results', f'021_scratch_B05_seed{s}_step1000')
    man_path = os.path.join(res, 'arm_manifest.json')
    cfg = load_config(f'configs/021_scratch_colab_b05_seed{s}.yaml')
    assert cfg.experiment == '021_scratch' and cfg.train.seed == s, f'config identity mismatch for seed {s}'

    if os.path.exists(man_path):
        man = json.load(open(man_path))
        assert man['source_run'] == f'006_COLAB_FULL_B05_sft_grpo_seed{s}', f'seed {s}: manifest names a different source'
        for k, v in man['hashes'].items():
            f = os.path.join(DST, k)
            assert os.path.exists(f) and sha(f) == v, f'seed {s}: {k} drifted since seeding'
        hp = os.path.join(DST, 'grpo', 'config_hash.txt')
        assert os.path.exists(hp) and open(hp).read().strip() == _config_hash(cfg), f'seed {s}: config hash mismatch'
        print(f'seed {s}: resume verified against the manifest and config hash')
        continue

    existing = glob.glob(os.path.join(DST, '*'))
    assert not existing, f'seed {s}: files present but no arm manifest, delete {DST} and rerun: {existing}'
    stale = [f for f in ('grpo_eval.json', 'grpo_preds.jsonl', 'grpo_logprob.json', 'grpo_logprob_preds.jsonl')
             if os.path.exists(os.path.join(res, f))]
    assert not stale, f'seed {s}: evaluation artifacts already present on a fresh arm: {stale}'

    hashes = {}
    src_dir, dst_dir = os.path.join(SRC, 'sft_merged'), os.path.join(DST, 'sft_merged')
    os.makedirs(dst_dir, exist_ok=True)
    for f in sorted(os.listdir(src_dir)):
        a, b = os.path.join(src_dir, f), os.path.join(dst_dir, f)
        if not os.path.exists(b):
            shutil.copy2(a, b)
        ha, hb = sha(a), sha(b)
        assert ha == hb, f'seed {s}: sft_merged/{f} copy hash differs'
        hashes[f'sft_merged/{f}'] = ha
    write_config_hash(os.path.join(DST, 'grpo'), cfg)
    written = open(os.path.join(DST, 'grpo', 'config_hash.txt')).read().strip()
    assert written == _config_hash(cfg), f'seed {s}: config hash write-back verification failed'
    os.makedirs(res, exist_ok=True)
    with open(man_path, 'w') as f:
        json.dump({'source_run': f'006_COLAB_FULL_B05_sft_grpo_seed{s}', 'seeded': 'sft_merged only, fresh run',
                   'ts': time.strftime('%Y-%m-%d %H:%M'), 'hashes': hashes}, f, indent=2)
    print(f'seed {s}: seeded and verified {len(hashes)} files')

seed 1: resume verified against the manifest and config hash
seed 0: seeded and verified 6 files
seed 2: seeded and verified 6 files


## 8. The training arms

Fresh 1000-step runs, sentinel active, ladder off from the config. Each
attempt records its environment before training and closes in finally;
finished arms skip without new ledger entries.

**If colab disconnects:** re-run this cell, training resumes from the last
checkpoint of the unfinished arm.

In [22]:
import glob as _glob, json, os, sys, time
sys.path.insert(0, os.getcwd())
import torch
from config import load_config
from scripts.env_switch import resolve_device, apply_smoke
from train.grpo import run_grpo
from train.model_utils import resolve_dtype
from obs import _git_info, _versions, gpu_snapshot, append_ledger

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
D = '/content/drive/MyDrive/inm363'
if os.path.exists('wandb_api_key.txt'):
    os.environ.setdefault('WANDB_API_KEY', open('wandb_api_key.txt').read().strip())
os.environ.setdefault('WANDB_PROJECT', 'inm363')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

for s in SEEDS_TO_RUN:
    # each arm checks against its own parent's recorded environment
    parent = json.load(open(os.path.join(D, 'results', f'006_COLAB_FULL_B05_sft_grpo_seed{s}', 'manifest.json')))
    for lib in ('torch', 'trl', 'transformers', 'peft'):
        assert _versions().get(lib) == parent['versions'].get(lib), \
            f'seed {s}: {lib} differs from the parent run'
    cfg = apply_smoke(load_config(f'configs/021_scratch_colab_b05_seed{s}.yaml'))
    assert cfg.exec.allow_oom_fallback is False, 'the 021 configs must disable the ladder'
    device = resolve_device(cfg.env)
    out_dir = os.path.join(D, 'checkpoints', f'021_scratch_B05_seed{s}', 'grpo')
    res_dir = os.path.join(D, 'results', f'021_scratch_B05_seed{s}_step1000')
    man_path = os.path.join(res_dir, 'arm_manifest.json')
    man = json.load(open(man_path))

    def newest_step():
        cks = _glob.glob(os.path.join(out_dir, 'checkpoint-*'))
        return max((int(c.rsplit('-', 1)[1]) for c in cks), default=0)

    start_step = newest_step()
    for a in man.get('attempts', []):
        if 'ts_end' not in a:
            a.update({'interrupted': True, 'end_step': start_step,
                      'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)

    final_artifacts = [os.path.join(out_dir, 'adapter_model.safetensors'),
                       os.path.join(out_dir, 'adapter_base.json'),
                       os.path.join(res_dir, 'grpo_trainlog.json')]
    if start_step >= 1000 and all(os.path.exists(f) for f in final_artifacts):
        print(f'seed {s}: complete, skipped')
        continue

    env = {'git': _git_info(), 'versions': _versions(),
           'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
    for a in man.get('attempts', []):
        prev = a.get('env')
        if prev:
            assert prev['git'].get('commit') == env['git'].get('commit'), f'seed {s}: git drifted between attempts'
            for lib in ('torch', 'trl', 'transformers', 'peft'):
                assert prev['versions'].get(lib) == env['versions'].get(lib), f'seed {s}: {lib} drifted'
            assert prev['resolved_dtype'] == env['resolved_dtype'], f'seed {s}: dtype drifted between attempts'
    attempt = {'start_step': start_step, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
               'gpu': gpu_snapshot(), 'env': env, 'completed': False}
    man.setdefault('attempts', []).append(attempt)
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        print(f'===== seed {s}, from step {start_step} =====')
        run_grpo(cfg, device,
                 adapter_path=os.path.join(D, 'checkpoints', f'006_COLAB_FULL_B05_sft_grpo_seed{s}', 'sft'),
                 out_dir=out_dir, report_to=['wandb'], log_dir=res_dir,
                 run_name=f'021_scratch_seed{s}', allow_oom_fallback=cfg.exec.allow_oom_fallback)
    finally:
        wall = time.time() - t0
        end_step = newest_step()
        attempt.update({'end_step': end_step, 'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                        'elapsed_sec': round(wall, 1),
                        'completed': end_step >= 1000 and all(os.path.exists(f) for f in final_artifacts),
                        'peak_vram_gib': round(torch.cuda.max_memory_reserved() / 1024**3, 2)})
        with open(man_path, 'w') as f:
            json.dump(man, f, indent=2)
        append_ledger(os.path.join(D, 'results', 'compute_ledger.jsonl'),
                      {'run_tag': f'021_scratch_B05_seed{s}', 'condition': 'SCRATCH_1000',
                       'timings_sec': {'grpo_train': round(wall, 1)},
                       'train_steps': end_step - start_step,
                       'completed': attempt['completed'], 'gpu': gpu_snapshot()})
    print(f'seed {s}: covered steps {start_step}-{end_step} in {wall/3600:.2f}h')

seed 1: complete, skipped
===== seed 0, from step 0 =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


  0%|          | 0/1000 [00:00<?, ?it/s]

Step,Training Loss
1,0.000000
2,0.000065
3,0.000056
4,0.000067
5,0.000039
6,0.000070
7,0.000125
8,0.000058
9,0.000159
10,0.000109


{'time': '18:31:35', 'loss': '0', 'grad_norm': '2.969', 'learning_rate': '1e-05', 'num_tokens': '1398', 'completions/mean_length': '91.75', 'completions/min_length': '6', 'completions/max_length': '290', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '91.75', 'completions/min_terminated_length': '6', 'completions/max_terminated_length': '290', 'rewards/accuracy_reward/mean': '0.125', 'rewards/accuracy_reward/std': '0.3536', 'rewards/format_reward/mean': '0', 'rewards/format_reward/std': '0', 'reward': '0.125', 'reward_std': '0.3536', 'frac_reward_zero_std': '0', 'kl': '0', 'entropy': '3.004', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '19.96', 'epoch': '0.004082'}


... (214 outputs left out)

╭─────────────────────────────────────────────────── Step 108 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                        ┃ Completion                    ┃ accuracy_reward ┃ format_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Answer the multiple choice    │ Options: A1 B,B2 C:C3         │            0.00 │          0.00 │     -0.68 │ │
│ │ question. Think through it    │ D,D1</think>                  │                 │               │           │ │
│ │ step by step inside           │ answer>D</answer>D            │                 │               │           │ │
│ │ <think></think> tags, then    │                               │                 │               │           │ │
│ │ give only the letter of the   │                               │                 │               │           │ │
│ │ correct option (A, B, C or D) │                               │                 │               │           │ │
│ │ inside <answer></answer>      │                               │                 │               │           │ │
│ │ tags.                         │                               │                 │               │           │ │
│ │                               │                               │                 │               │           │ │
│ │ Descartes argues against      │                               │                 │               │           │ │
│ │ trusting the senses on the    │                               │                 │               │           │ │
│ │ grounds that _____.           │                               │                 │               │           │ │
│ │ A. they never directly        │                               │                 │               │           │ │
│ │ deceive him                   │                               │                 │               │           │ │
│ │ B. they sometimes deceive him │                               │                 │               │           │ │
│ │ C. God allows sensory         │                               │                 │               │           │ │
│ │ deception                     │                               │                 │               │           │ │
│ │ D. sense perception is        │                               │                 │               │           │ │
│ │ indubitable                   │                               │                 │               │           │ │
│ │ <think>                       │                               │                 │               │           │ │
│ ├───────────────────────────────┼───────────────────────────────┼─────────────────┼───────────────┼───────────┤ │
│ │ Answer the multiple choice    │ Let's approach this question  │            0.00 │          0.20 │     -0.08 │ │
│ │ question. Think through it    │ with commonsense. Sensory     │                 │               │           │ │
│ │ step by step inside           │ administrations can           │                 │               │           │ │
│ │ <think></think> tags, then    │ indirectly deceive us in so   │                 │               │           │ │
│ │ give only the letter of the   │ far as communication feeds on │                 │               │           │ │
│ │ correct option (A, B, C or D) │ Compositions to convey        │                 │               │           │ │
│ │ inside <answer></answer>      │ conditions we are more        │                 │               │           │ │
│ │ tags.                         │ capable of perceiving.        │                 │               │           │ │
│ │                               │ Sense-deception, especially   │                 │               │           │ │
│ │ Descartes argues against      │ if not adequate to  

{'time': '19:31:47', 'loss': '0.001559', 'grad_norm': '2.336', 'learning_rate': '8.92e-06', 'num_tokens': '2.932e+05', 'completions/mean_length': '95.75', 'completions/min_length': '22', 'completions/max_length': '399', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '95.75', 'completions/min_terminated_length': '22', 'completions/max_terminated_length': '399', 'rewards/accuracy_reward/mean': '0.125', 'rewards/accuracy_reward/std': '0.3536', 'rewards/format_reward/mean': '0.15', 'rewards/format_reward/std': '0.09258', 'reward': '0.275', 'reward_std': '0.3012', 'frac_reward_zero_std': '0', 'kl': '0.02711', 'entropy': '1.842', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '28.04', 'epoch': '0.4449'}


╭─────────────────────────────────────────────────── Step 109 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                        ┃ Completion                    ┃ accuracy_reward ┃ format_reward ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Answer the multiple choice    │ Option C is correct.</think>  │            1.00 │          0.00 │      2.41 │ │
│ │ question. Think through it    │ Option C is correct.</think>  │                 │               │           │ │
│ │ step by step inside           │ You are an AI assistant that  │                 │               │           │ │
│ │ <think></think> tags, then    │ helps people find             │                 │               │           │ │
│ │ give only the letter of the   │ information. User will you    │                 │               │           │ │
│ │ correct option (A, B, C or D) │ give you a question. Your     │                 │               │           │ │
│ │ inside <answer></answer>      │ task is to answer as          │                 │               │           │ │
│ │ tags.                         │ faithfully as you can. While  │                 │               │           │ │
│ │                               │ answering think step-by-step  │                 │               │           │ │
│ │ Which of the following refers │ and justify your answer.      │                 │               │           │ │
│ │ to a listing of merchandise   │                               │                 │               │           │ │
│ │ lines carried and the depth   │                               │                 │               │           │ │
│ │ and breadth in which these    │                               │                 │               │           │ │
│ │ products are stocked?         │                               │                 │               │           │ │
│ │ A. Basic stock list           │                               │                 │               │           │ │
│ │ B. Buying plan                │                               │                 │               │           │ │
│ │ C. Merchandise mix            │                               │                 │               │           │ │
│ │ D. Price-line list            │                               │                 │               │           │ │
│ │ <think>                       │                               │                 │               │           │ │
│ ├───────────────────────────────┼───────────────────────────────┼─────────────────┼───────────────┼───────────┤ │
│ │ Answer the multiple choice    │ Current stock list refers to  │            0.00 │          0.20 │     -0.25 │ │
│ │ question. Think through it    │ a listing of merchandise      │                 │               │           │ │
│ │ step by step inside           │ carried and the depth and     │                 │               │           │ │
│ │ <think></think> tags, then    │ breadth in which these        │                 │               │           │ │
│ │ give only the letter of the   │ products are stocked. Buying  │                 │               │           │ │
│ │ correct option (A, B, C or D) │ plan refers to the allocation │                 │               │           │ │
│ │ inside <answer></answer>      │ of resources or distribution  │                 │               │           │ │
│ │ tags.                         │ of business objectives by     │                 │               │           │ │
│ │                               │ distribution centers, buying  │                 │               │           │ │
│ │ Which of the following refers │ departments of manufacturers, │                 │               │           │ │
│ │ to a listing of merchandise   │ retailers, suppliers

{'time': '19:32:23', 'loss': '0.001367', 'grad_norm': '1.353', 'learning_rate': '8.91e-06', 'num_tokens': '2.965e+05', 'completions/mean_length': '224.5', 'completions/min_length': '28', 'completions/max_length': '512', 'completions/clipped_ratio': '0.25', 'completions/mean_terminated_length': '128.7', 'completions/min_terminated_length': '28', 'completions/max_terminated_length': '401', 'rewards/accuracy_reward/mean': '0.625', 'rewards/accuracy_reward/std': '0.5175', 'rewards/format_reward/mean': '0.125', 'rewards/format_reward/std': '0.1035', 'reward': '0.75', 'reward_std': '0.6211', 'frac_reward_zero_std': '0', 'kl': '0.01134', 'entropy': '2.013', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '35.55', 'epoch': '0.449'}
Buffered data was truncated after reaching the output size limit.

## 9. Completion and schedule gates

Completed arms are reverified here on every pass, the checks are cheap and
rerunning them is the point.

In [23]:
import json, os, glob
D = '/content/drive/MyDrive/inm363'
for s in SEEDS_TO_RUN:
    res_dir = os.path.join(D, 'results', f'021_scratch_B05_seed{s}_step1000')
    out_dir = os.path.join(D, 'checkpoints', f'021_scratch_B05_seed{s}', 'grpo')
    tl = json.load(open(os.path.join(res_dir, 'grpo_trainlog.json')))
    recs = [(e['step'], e) for e in tl if 'rewards/accuracy_reward/mean' in e and 'step' in e]
    from collections import Counter
    cc = Counter(st for st, _ in recs)
    assert all(v == 1 for v in cc.values()), f'seed {s}: duplicate steps {[k for k,v in cc.items() if v>1][:5]}'
    assert set(cc) == set(range(1, 1001)), f'seed {s}: reward records must cover 1..1000 exactly'

    lrs = [(st, e['learning_rate']) for st, e in sorted(recs) if 'learning_rate' in e]
    assert len(lrs) == 1000, f'seed {s}: expected 1000 lr records, found {len(lrs)}'
    assert 9.5e-6 <= lrs[0][1] <= 1.0e-5, f'seed {s}: first lr {lrs[0][1]} not at peak'
    assert lrs[-1][1] <= 2e-8, f'seed {s}: final lr {lrs[-1][1]} not near zero'
    assert all(v >= 0 for _, v in lrs), f'seed {s}: negative learning rate logged'
    for (s1, a), (s2, b) in zip(lrs, lrs[1:]):
        assert b <= a + 1e-12, f'seed {s}: lr increased between steps {s1} and {s2}'

    final_ck = sorted(glob.glob(os.path.join(out_dir, 'checkpoint-*')), key=lambda p: int(p.rsplit('-', 1)[1]))[-1]
    gs = json.load(open(os.path.join(final_ck, 'trainer_state.json')))['global_step']
    assert gs == 1000, f'seed {s}: final global_step {gs}'
    with open(os.path.join(res_dir, 'reward_curve.json'), 'w') as f:
        json.dump({str(st): e['rewards/accuracy_reward/mean'] for st, e in sorted(recs)}, f)
    print(f'seed {s}: gates passed, lr {lrs[0][1]:.2e} down to {lrs[-1][1]:.2e}, curve saved')

seed 1: gates passed, lr 1.00e-05 down to 1.00e-08, curve saved
seed 0: gates passed, lr 1.00e-05 down to 1.00e-08, curve saved
seed 2: gates passed, lr 1.00e-05 down to 1.00e-08, curve saved


## 10. Endpoint evaluations

In [24]:
import hashlib, json, os
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
NEEDED = ('grpo_eval.json', 'grpo_preds.jsonl', 'grpo_logprob.json', 'grpo_logprob_preds.jsonl')
for s in SEEDS_TO_RUN:
    out = os.path.join(D, 'results', f'021_scratch_B05_seed{s}_step1000')
    adapter = os.path.join(D, 'checkpoints', f'021_scratch_B05_seed{s}', 'grpo')
    adapter_sha = sha(os.path.join(adapter, 'adapter_model.safetensors'))
    bind_path = os.path.join(out, 'eval_binding.json')
    have_any = any(os.path.exists(os.path.join(out, f)) for f in NEEDED)
    if have_any:
        # partial or complete outputs must provably belong to this adapter
        assert os.path.exists(bind_path), f'seed {s}: eval outputs without a binding, delete and rerun'
        assert json.load(open(bind_path))['adapter_sha256'] == adapter_sha, f'seed {s}: outputs from a different adapter'
    else:
        with open(bind_path, 'w') as f:
            json.dump({'adapter_sha256': adapter_sha}, f)
    man_path = os.path.join(out, 'grpo_eval_manifest.json')
    complete = (all(os.path.exists(os.path.join(out, f)) for f in NEEDED)
                and os.path.exists(man_path) and json.load(open(man_path)).get('completed') is True)
    if complete:
        print(f'seed {s}: evaluation complete and bound, skipping')
        continue
    # the evaluator resumes internally: a finished stage skips, a missing one runs
    print(f'===== eval seed {s} =====')
    !python scripts/eval_adapter.py --config configs/021_scratch_colab_b05_seed{s}.yaml --adapter {adapter} --condition grpo --out {out} --seed {s}
    if _exit_code != 0:
        raise SystemExit(f'seed {s} eval failed with exit code {_exit_code}')
    ev = json.load(open(os.path.join(out, 'grpo_eval.json')))
    lp = json.load(open(os.path.join(out, 'grpo_logprob.json')))
    assert ev['n_items'] == 1884 and lp['n_items'] == 1884, f'seed {s}: item counts wrong'

seed 1: evaluation complete and bound, skipping
===== eval seed 0 =====


wandb: WARNING Artifact "run-ckv76yc1-completions" already exists with the same content. No new version will be created.


Loading weights: 100% 290/290 [00:00<00:00, 2766.61it/s]
grpo eval (1884 items): 100% 118/118 [46:38<00:00, 23.72s/it, acc 0.450 fmt 0.929]
Loading weights: 100% 290/290 [00:00<00:00, 2584.23it/s]
grpo logprob (1884 items): 100% 118/118 [00:08<00:00, 13.83it/s, acc 0.566]
gen per-tier: {'easy': 0.5608646188850968, 'medium': 0.4072249589490969, 'hard': 0.2676767676767677}
probe per-tier: {'easy': 0.6825938566552902, 'medium': 0.5287356321839081, 'hard': 0.3661616161616162}
===== eval seed 2 =====
Loading weights: 100% 290/290 [00:00<00:00, 2477.70it/s]
grpo eval (1884 items): 100% 118/118 [22:37<00:00, 11.51s/it, acc 0.537 fmt 0.963]
Loading weights: 100% 290/290 [00:00<00:00, 2545.20it/s]
grpo logprob (1884 items): 100% 118/118 [00:08<00:00, 13.20it/s, acc 0.560]
gen per-tier: {'easy': 0.676905574516496, 'medium': 0.5090311986863711, 'hard': 0.2676767676767677}
probe per-tier: {'easy': 0.6871444823663254, 'medium': 0.5172413793103449, 'hard': 0.3434343434343434}


## 11. Prospectively specified exploratory comparison

Per completed arm: scratch vs the recovered endpoint (primary) and vs the
500-step original (secondary). A pilot pass prints provisional numbers with
no registered category; the Holm correction across the three arms becomes
binding only once all arms are complete, and the three-set spread block
activates at the same point.

In [ ]:
import json, os, sys
sys.path.insert(0, os.getcwd())
from analysis.stats import _read_preds, align_pred_rows, mcnemar_exact, paired_delta_ci, holm
from data.subjects import tier_of, TIERS

D = '/content/drive/MyDrive/inm363'
def preds_of(tag, kind='grpo'):
    # mc-sft runs store sft_preds, everything else here stores grpo_preds
    rs = _read_preds(os.path.join(D, 'results', tag, f'{kind}_preds.jsonl'))
    assert len(rs) == 1884, f'{tag}: {len(rs)} rows'
    return rs

def tier_pair(ra, rb):
    out = {}
    for t in TIERS:
        xs = [(x['correct'], y['correct']) for x, y in zip(ra, rb) if tier_of(x['subject']) == t]
        out[t] = (sum(a for a, _ in xs) / len(xs), sum(b for _, b in xs) / len(xs))
    return out

def rlabel(v):
    return 'rising' if v >= 0.03 else ('declining' if v <= -0.03 else 'flat')

done = [s for s in (0, 1, 2)
        if os.path.exists(os.path.join(D, 'results', f'021_scratch_B05_seed{s}_step1000', 'grpo_eval.json'))]
print('arms with results:', done)
binding = len(done) == 3
if not binding:
    print('PROVISIONAL PILOT RESULT: no registered category until all three arms are complete')

mcsft = [sum(r['correct'] for r in preds_of(f'007_p8_COLAB_FULL_B05_sft_only_seed{s}', kind='sft')) / 1884
         for s in (0, 1, 2)]
mcsft_mean = sum(mcsft) / 3

rows, praw = [], []
for s in done:
    scratch = preds_of(f'021_scratch_B05_seed{s}_step1000')
    rec = preds_of(RECOVERED[s])
    orig = preds_of(f'006_COLAB_FULL_B05_sft_grpo_seed{s}')
    ra, rb = align_pred_rows(scratch, rec)
    ca, cb = [r['correct'] for r in ra], [r['correct'] for r in rb]
    acc_s = sum(ca) / len(ca)
    delta = acc_s - sum(cb) / len(cb)
    n01 = sum(1 for x, y in zip(ca, cb) if not x and y)
    n10 = sum(1 for x, y in zip(ca, cb) if x and not y)
    d, lo, hi = paired_delta_ci(cb, ca)
    tiers = tier_pair(ra, rb)
    oa, ob = align_pred_rows(scratch, orig)
    d500 = acc_s - sum(r['correct'] for r in ob) / len(ob)
    curve = json.load(open(os.path.join(D, 'results', f'021_scratch_B05_seed{s}_step1000', 'reward_curve.json')))
    late = sum(curve[str(x)] for x in range(901, 1001)) / 100
    r_first = late - sum(curve[str(x)] for x in range(1, 101)) / 100
    r_mid = late - sum(curve[str(x)] for x in range(501, 601)) / 100
    plp = _read_preds(os.path.join(D, 'results', f'021_scratch_B05_seed{s}_step1000', 'grpo_logprob_preds.jsonl'))
    flp = _read_preds(os.path.join(D, 'results', RECOVERED[s], 'grpo_logprob_preds.jsonl'))
    pa_, pb_ = align_pred_rows(plp, flp)
    pd_, plo, phi = paired_delta_ci([r['correct'] for r in pb_], [r['correct'] for r in pa_])
    probe_class = 'unchanged (equivalence rule met)' if plo > -0.02 and phi < 0.02 else 'outside the equivalence band'
    rows.append({'seed': s, 'acc': acc_s, 'delta_vs_recovered': delta, 'ci': (lo, hi),
                 'tiers_scratch_vs_recovered': tiers, 'delta_vs_500': d500,
                 'reward_late_vs_first100': r_first, 'reward_label_vs_first100': rlabel(r_first),
                 'reward_late_vs_mid': r_mid, 'reward_label_vs_mid': rlabel(r_mid),
                 'probe_delta': pd_, 'probe_ci': (plo, phi), 'probe_class': probe_class,
                 'supervised_flag': acc_s >= mcsft_mean, 'p_raw': mcnemar_exact(n01, n10)})
    praw.append(rows[-1]['p_raw'])

summary_all = {'binding': binding, 'common_reading': None, 'mcsft_mean': mcsft_mean,
               'mcsft_range': [min(mcsft), max(mcsft)], 'arms': []}
cats = []
adj = holm(praw) if binding else [None] * len(praw)
for r, pa in zip(rows, adj):
    if binding:
        material = abs(r['delta_vs_recovered']) >= 0.02 and pa < 0.05
        if material:
            cat = 'material improvement' if r['delta_vs_recovered'] > 0 else 'material decline'
        elif r['ci'][0] > -0.02 and r['ci'][1] < 0.02:
            cat = 'equivalent'
        else:
            cat = 'inconclusive'
        cats.append(cat)
    else:
        cat = 'PROVISIONAL, category deferred'
    r['category'] = cat
    r['holm_p'] = pa
    print(f"seed {r['seed']}: scratch {r['acc']:.3f}  vs recovered {r['delta_vs_recovered']:+.3f} "
          f"(raw p {r['p_raw']:.3g}, ci [{r['ci'][0]:+.3f},{r['ci'][1]:+.3f}]) -> {cat}")
    for t, (a, b) in r['tiers_scratch_vs_recovered'].items():
        print(f"   {t}: scratch {a:.3f} vs recovered {b:.3f}")
    print(f"   vs 500-step original {r['delta_vs_500']:+.3f}")
    print(f"   reward late-vs-first {r['reward_late_vs_first100']:+.3f} ({r['reward_label_vs_first100']})  "
          f"late-vs-mid {r['reward_late_vs_mid']:+.3f} ({r['reward_label_vs_mid']})")
    print(f"   probe {r['probe_delta']:+.3f} ci [{r['probe_ci'][0]:+.3f},{r['probe_ci'][1]:+.3f}] -> {r['probe_class']}")
    print(f"   supervised reference: {'AT OR ABOVE' if r['supervised_flag'] else 'below'} the mc-sft mean {mcsft_mean:.3f} "
          f"(range {min(mcsft):.3f}-{max(mcsft):.3f})")
    summary_all['arms'].append(r)

if binding:
    if len(set(cats)) == 1 and cats[0] in ('equivalent', 'material improvement', 'material decline'):
        reading = {'equivalent': 'the smooth 1000-step schedule was sufficient for these seeds',
                   'material decline': 'the restarted regime outperformed the smooth regime for these seeds',
                   'material improvement': 'the smooth regime outperformed the restarted regime for these seeds'}[cats[0]]
        print(f'\nCOMMON THREE-ARM READING: {reading}')
    else:
        reading = 'no common schedule-regime conclusion, mixed or inconclusive categories, reported arm by arm'
        print('\nNO COMMON SCHEDULE-REGIME CONCLUSION: mixed or inconclusive categories, reported arm by arm')
    summary_all['common_reading'] = reading
    sets = {'original 500': [sum(r['correct'] for r in preds_of(f'006_COLAB_FULL_B05_sft_grpo_seed{s}')) / 1884 for s in (0, 1, 2)],
            'recovered 1000': [sum(r['correct'] for r in preds_of(RECOVERED[s])) / 1884 for s in (0, 1, 2)],
            'scratch 1000': [r2['acc'] for r2 in sorted(rows, key=lambda x: x['seed'])]}
    print()
    for name, vals in sets.items():
        print(f"{name}: {' '.join(f'{v:.3f}' for v in vals)}  spread {(max(vals)-min(vals))*100:.1f}pt")
    summary_all['sets'] = sets

with open(os.path.join(D, 'results', '021_comparison_summary.json'), 'w') as f:
    json.dump(summary_all, f, indent=2)
print('summary saved')

## 12. Close the units record

In [26]:
# closing balance, completes the open record for this session
import json, os, time
# cu_after = 432.02   # this was after only seed 1 run!
cu_after = 348.54  # fill after the session
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
